# v1(김도엽) 앙상블에 핫스팟 피처 얹어보기

- 목적: 도엽님 v1(CV 0.4401)에 김지연 핫스팟 피처(N≥10)를 더하면 점수가 오르는지 확인함
- v1 구성: LogisticRegression + SGDClassifier + XGBoost 확률 평균 (+ 클래스별 편향 조정)
- 시간을 줄이려고 Optuna 튜닝은 다시 하지 않고, v1에서 찾은 하이퍼파라미터(`meta.json`)를 그대로 씀
- 비교 방법: 같은 5-Fold에서 **핫스팟 없음 vs 있음**을 나란히 돌려 Macro F1을 비교함

## Data Leakage 방지

- 변이 사전(`변이 표기`, `핫스팟`)과 변이 수 표준화는 **fold마다 학습 부분으로만** 만듦
- 핫스팟은 정답(SUBCLASS)을 보지 않고 **빈도(10명 이상)** 로만 고름
- test는 이 노트북에서 쓰지 않음 (비교 실험용)

## 1. 환경 설정

In [ ]:
import os                     # 결과 폴더 생성용
import re                     # 변이 표기에서 숫자(위치)를 찾는 도구
import time                   # fold마다 걸린 시간 확인용
import numpy as np            # 수치 계산
import pandas as pd           # 표 데이터 처리
import scipy.sparse as sp     # 0이 대부분인 큰 표를 가볍게 다루는 희소 행렬
from sklearn.feature_extraction.text import CountVectorizer        # 토큰 목록 → 0/1 열로 바꿔줌
from sklearn.linear_model import LogisticRegression, SGDClassifier # v1의 선형 모델 2개
from sklearn.model_selection import StratifiedKFold                # 클래스 비율을 유지하는 K-Fold
from sklearn.metrics import f1_score, classification_report        # 평가 지표
from sklearn.preprocessing import LabelEncoder                     # 아형 이름 → 숫자
from sklearn.utils.class_weight import compute_sample_weight       # XGBoost 불균형 보정용
from xgboost import XGBClassifier                                  # v1의 트리 모델

DATA_DIR = './data'   # 데이터 폴더
SEED = 42             # v1과 같은 시드 (fold 나누는 방식도 같아짐)
HOT_N = 10            # 핫스팟 기준: train fold에서 N명 이상

## 2. 데이터 불러오기

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', dtype=str)          # 모든 값을 글자로 읽음 (변이 표기 보존)
genes = [c for c in train.columns if c not in ('ID', 'SUBCLASS')] # 유전자 컬럼 4,384개
V = train[genes].fillna('WT').values                              # 빈 칸은 WT로 (train에는 빈 칸 없음)
G = np.array(genes)                                               # 열 번호 → 유전자 이름

le = LabelEncoder()                     # 아형 이름을 0~25로 바꾸는 인코더
y = le.fit_transform(train['SUBCLASS']) # train 라벨로만 fit 함
print(V.shape, '클래스 수:', len(le.classes_))

## 3. 환자별 토큰과 변이 수 만들기

v1과 같은 피처를 만듦. 모두 **환자 한 명의 행 안에서만** 계산하는 값이라 여기서는 leakage가 없음.
- **변이 표기 토큰**: `IDH1=R132H`처럼 유전자+변이 표기 (한 칸에 여러 개면 공백으로 나눔)
- **핫스팟 토큰**: `IDH1_132`처럼 유전자+위치 (같은 위치의 R132H, R132C가 하나로 묶임)
- **변이 수(burden) 14개**: 변이 유전자 수, 변이 표기 수, 비율, 여러 변이가 든 칸 수, 변이 유형별 개수·비율

In [ ]:
MISSENSE = re.compile(r'^([A-Z])(\d+)([A-Z])$')   # 'R132H'처럼 글자-숫자-글자 모양

def variant_type(tok):
    # 변이 표기 모양으로 대략 나눔 (v1과 같은 규칙, 기능을 확정하는 건 아님)
    if 'fs' in tok:                                  # frameshift (읽는 틀이 밀림)
        return 0
    if '*' in tok:                                   # nonsense (중간에 끊김)
        return 1
    m = MISSENSE.match(tok)
    if m:
        return 2 if m.group(1) == m.group(3) else 3  # 2: 아미노산 그대로(silent), 3: 바뀜(missense)
    return 4                                         # 그 밖의 표기

def parse_rows(V):
    tok_docs, hot_docs = [], []                 # 환자별 변이 표기 토큰, 핫스팟 토큰
    B = np.zeros((len(V), 14))                  # 환자별 변이 수 14개
    for i, row in enumerate(V):                 # 환자 한 명씩
        idx = np.flatnonzero(row != 'WT')       # 변이가 있는 유전자 위치
        toks, hots, all_muts = [], [], []
        for j in idx:
            muts = row[j].split()               # 'Q369* I368N' → ['Q369*', 'I368N']
            all_muts += muts
            toks.extend(dict.fromkeys(f'{G[j]}={m}' for m in muts))   # 중복 제거하며 'TP53=R273H' 모양
            for m in muts:
                nums = re.findall(r'\d+', m)   # 변이 표기 속 숫자
                if nums:
                    hots.append(f'{G[j]}_{nums[0]}')                  # 첫 숫자가 위치 → 'TP53_273'
        tok_docs.append(toks)
        hot_docs.append(list(dict.fromkeys(hots)))                    # 같은 위치는 한 번만
        types = [variant_type(m) for m in all_muts]
        cnt = [types.count(k) for k in range(5)]                      # 유형별 개수
        B[i, :4] = [len(idx), len(all_muts), len(idx) / V.shape[1], sum(' ' in row[j] for j in idx)]
        B[i, 4:9] = cnt
        B[i, 9:] = [c / len(all_muts) if all_muts else 0 for c in cnt]   # 유형별 비율
    return tok_docs, hot_docs, np.log1p(B)      # 변이 수는 log1p로 눌러줌 (v1과 같음)

t0 = time.time()
TOK, HOT, BUR = parse_rows(V)
BIN = sp.csr_matrix((V != 'WT').astype(np.float32))   # 유전자별 변이 있음 1 / 없음 0
print(f'완료 {time.time() - t0:.1f}초 | 예시 토큰: {TOK[0][:3]} | 예시 핫스팟: {HOT[0][:3]}')

## 4. 모델 정의 (v1 튜닝 결과 그대로)

- 선형 모델(LR, SGD) 입력: 0/1 + 변이 표기 토큰 + 변이 수(표준화)
- XGBoost 입력: 0/1 + 변이 수 → 여기에 **핫스팟을 더한 버전**과 비교함
- v1의 XGBoost는 변이 표기 토큰을 안 쓰기 때문에, 위치 정보를 처음 받게 되는 곳이 바로 XGBoost임

In [ ]:
def make_lr():
    return LogisticRegression(C=0.1525, class_weight='balanced', max_iter=3000)

def make_sgd():
    return SGDClassifier(loss='log_loss', alpha=0.000665, penalty='elasticnet', l1_ratio=0.4138,
                         class_weight='balanced', max_iter=50, tol=None, random_state=SEED)

def make_xgb():
    return XGBClassifier(n_estimators=200, learning_rate=0.0458, max_depth=4, subsample=0.81,
                         colsample_bytree=0.316, reg_lambda=0.382, reg_alpha=0.0685,
                         tree_method='hist', n_jobs=-1, random_state=SEED)

identity = lambda doc: doc   # 토큰 목록을 그대로 쓰라는 뜻 (CountVectorizer가 글자를 쪼개지 않게)

## 5. 교차검증 (5-Fold) - 핫스팟 없음 vs 있음

- 모델 4개(LR, SGD, XGB, XGB+핫스팟)의 검증 확률(OOF)을 모아둠
- LR, SGD는 두 버전에서 같으므로 한 번만 학습함
- 걸리는 시간: fold당 약 1분 (XGBoost 2번 학습)

In [ ]:
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(V, y))
K = len(le.classes_)
P = {name: np.zeros((len(y), K)) for name in ['LR', 'SGD', 'XGB', 'XGB_hot']}   # OOF 확률 모음

for f, (tr, va) in enumerate(folds, 1):
    t0 = time.time()
    # 사전은 학습 fold로만 만듦 → 검증 fold에만 있는 변이는 무시됨
    tok_vec = CountVectorizer(analyzer=identity, min_df=2, binary=True, lowercase=False).fit([TOK[i] for i in tr])
    hot_vec = CountVectorizer(analyzer=identity, min_df=HOT_N, binary=True, lowercase=False).fit([HOT[i] for i in tr])
    mu, sd = BUR[tr].mean(0), BUR[tr].std(0) + 1e-9    # 변이 수 표준화도 학습 fold 통계만

    def linear_X(ix):   # LR·SGD용 입력
        return sp.hstack([BIN[ix], tok_vec.transform([TOK[i] for i in ix]),
                          sp.csr_matrix((BUR[ix] - mu) / sd)], format='csr').astype(np.float32)

    def tree_X(ix, use_hot):   # XGBoost용 입력 (use_hot=True면 핫스팟 열 추가)
        parts = [BIN[ix], sp.csr_matrix(BUR[ix])]
        if use_hot:
            parts.append(hot_vec.transform([HOT[i] for i in ix]))
        return sp.hstack(parts, format='csr').astype(np.float32)

    Ltr, Lva = linear_X(tr), linear_X(va)
    P['LR'][va] = make_lr().fit(Ltr, y[tr]).predict_proba(Lva)
    P['SGD'][va] = make_sgd().fit(Ltr, y[tr]).predict_proba(Lva)

    sw = compute_sample_weight('balanced', y[tr])     # 학습 fold 기준 불균형 가중치
    for name, use_hot in [('XGB', False), ('XGB_hot', True)]:
        model = make_xgb().fit(tree_X(tr, use_hot), y[tr], sample_weight=sw)
        P[name][va] = model.predict_proba(tree_X(va, use_hot))

    print(f'fold {f} 완료 ({time.time() - t0:.0f}초) | 핫스팟 {len(hot_vec.vocabulary_)}개')

## 6. 결과 비교

In [ ]:
def score(prob):
    pooled = f1_score(y, prob.argmax(1), average='macro')                                  # 전체를 모아 한 번에
    by_fold = np.mean([f1_score(y[va], prob[va].argmax(1), average='macro') for _, va in folds])  # fold별 평균
    return round(pooled, 4), round(by_fold, 4)

P['ENS'] = (P['LR'] + P['SGD'] + P['XGB']) / 3          # v1 앙상블 (편향 조정 전)
P['ENS_hot'] = (P['LR'] + P['SGD'] + P['XGB_hot']) / 3  # XGBoost만 핫스팟 버전으로 바꾼 앙상블

result = pd.DataFrame([(name, *score(P[name])) for name in P], columns=['모델', 'CV(전체)', 'CV(fold 평균)'])
display(result)

In [ ]:
# 아형별 F1 비교: 핫스팟이 어떤 아형을 올리고 내렸는지
f1_base = f1_score(y, P['ENS'].argmax(1), average=None)
f1_hot = f1_score(y, P['ENS_hot'].argmax(1), average=None)
diff = pd.DataFrame({'아형': le.classes_, '핫스팟 없음': f1_base.round(3), '핫스팟 있음': f1_hot.round(3)})
diff['차이'] = (diff['핫스팟 있음'] - diff['핫스팟 없음']).round(3)
display(diff.sort_values('차이'))

## 7. 클래스별 편향 조정 (v1과 같은 방법)

- 확률에 log를 씌운 뒤, 아형마다 더할 값(편향)을 Macro F1이 오르는 쪽으로 조금씩 찾음
- 같은 데이터로 찾고 같은 데이터로 채점하면 점수가 부풀려짐 → **4개 fold로 편향을 찾고 남은 1개 fold로 채점**을 5번 반복함 (v1의 "Fold 교차" 평가)

In [ ]:
def macro_f1(y_true, S):
    return f1_score(y_true, S.argmax(1), average='macro')   # 점수(S)가 가장 큰 아형을 예측으로

def fit_bias(prob, y_true, steps=(-1.0, -0.5, -0.25, 0.25, 0.5, 1.0), n_rounds=2):
    L = np.log(prob + 1e-9)                    # 확률 → log 확률
    b = np.zeros(prob.shape[1])                # 아형별 편향, 처음엔 0
    best = macro_f1(y_true, L + b)
    for _ in range(n_rounds):                  # 전체를 2바퀴 돌며
        for c in range(prob.shape[1]):         # 아형 하나씩
            for s in steps:                    # 이만큼 더해보고
                b2 = b.copy(); b2[c] += s
                v = macro_f1(y_true, L + b2)
                if v > best + 1e-6:            # 점수가 오르면 채택
                    best, b = v, b2
    return b

fold_id = np.zeros(len(y), dtype=int)          # 환자마다 몇 번 fold의 검증이었는지
for k, (_, va) in enumerate(folds):
    fold_id[va] = k

def bias_cv(prob):
    scores = []
    for k in range(len(folds)):
        tr_m, va_m = fold_id != k, fold_id == k              # 나머지 4개 fold / 채점할 1개 fold
        b = fit_bias(prob[tr_m], y[tr_m])                    # 4개 fold로만 편향을 찾음
        scores.append(macro_f1(y[va_m], np.log(prob[va_m] + 1e-9) + b))
    return np.mean(scores), np.std(scores), np.round(scores, 4)

for name in ['ENS', 'ENS_hot']:
    m, s, fs = bias_cv(P[name])
    print(f'{name:8s} 편향 조정 전 {score(P[name])[1]:.4f} → 편향 조정 후 {m:.4f} ± {s:.4f} | fold별 {fs}')

## 8. 전체 train으로 최종 학습

- 비교 결과 더 좋은 구성(**LR + SGD + XGBoost(핫스팟) + 편향 조정**)으로 train 6,201명 전체를 학습함
- 변이 사전, 핫스팟 목록, 변이 수 표준화 모두 **train 전체로만** 만듦. test는 다음 단계에서 변환에만 씀
- 편향은 5-Fold OOF 확률 전체로 찾음 (v1과 같음)

In [ ]:
tok_vec = CountVectorizer(analyzer=identity, min_df=2, binary=True, lowercase=False).fit(TOK)
hot_vec = CountVectorizer(analyzer=identity, min_df=HOT_N, binary=True, lowercase=False).fit(HOT)
mu, sd = BUR.mean(0), BUR.std(0) + 1e-9
print('변이 표기 토큰:', len(tok_vec.vocabulary_), '/ 핫스팟:', len(hot_vec.vocabulary_))

def linear_full(B, T, Bu):   # LR·SGD용 입력 (train 전체 기준 사전·표준화)
    return sp.hstack([B, tok_vec.transform(T), sp.csr_matrix((Bu - mu) / sd)], format='csr').astype(np.float32)

def tree_full(B, H, Bu):     # XGBoost(핫스팟)용 입력
    return sp.hstack([B, sp.csr_matrix(Bu), hot_vec.transform(H)], format='csr').astype(np.float32)

L_all = linear_full(BIN, TOK, BUR)
final_lr = make_lr().fit(L_all, y)
final_sgd = make_sgd().fit(L_all, y)
final_xgb = make_xgb().fit(tree_full(BIN, HOT, BUR), y, sample_weight=compute_sample_weight('balanced', y))

final_bias = fit_bias(P['ENS_hot'], y)          # OOF 확률로 아형별 편향을 찾음
print('편향:', dict(zip(le.classes_, final_bias)))

## 9. test 예측 및 제출 파일

- test에만 빈 칸이 237개 있음 → train과 같은 규칙으로 WT 처리함 (통계를 쓰지 않으므로 leakage 아님)
- test는 위에서 train으로 만든 사전으로 **변환만** 하고, 아무것도 fit 하지 않음

In [ ]:
test = pd.read_csv(f'{DATA_DIR}/test.csv', dtype=str)
submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
V_test = test.reindex(columns=genes, fill_value='WT').fillna('WT').values   # train 유전자 순서로 맞추고 빈 칸은 WT
TOK_t, HOT_t, BUR_t = parse_rows(V_test)                                    # 행 단위 계산 (fit 없음)
BIN_t = sp.csr_matrix((V_test != 'WT').astype(np.float32))

L_t = linear_full(BIN_t, TOK_t, BUR_t)
prob_test = (final_lr.predict_proba(L_t) + final_sgd.predict_proba(L_t)
             + final_xgb.predict_proba(tree_full(BIN_t, HOT_t, BUR_t))) / 3   # 3개 모델 확률 평균
pred = (np.log(prob_test + 1e-9) + final_bias).argmax(1)                      # 편향 더한 뒤 가장 큰 아형

submission['SUBCLASS'] = le.inverse_transform(pred)
os.makedirs('./output', exist_ok=True)
submission.to_csv('./output/submission_v3_v1hot.csv', index=False, encoding='utf-8-sig')

# 제출 파일 점검
chk = pd.read_csv('./output/submission_v3_v1hot.csv')
print('행 수 일치:', len(chk) == len(test), '/ ID 순서 일치:', (chk['ID'] == test['ID']).all(),
      '/ 결측 없음:', chk['SUBCLASS'].notna().all(), '/ 예측된 아형 수:', chk['SUBCLASS'].nunique())
print(chk['SUBCLASS'].value_counts().head(10))

## 7. 해석 메모

- v1 원래 점수 0.4401은 위 앙상블에 **클래스별 편향 조정**(+0.004 정도)을 더한 값이라, 여기 숫자와 바로 비교하지 않음. **같은 노트북 안의 "핫스팟 없음 vs 있음" 차이**만 봄
- fold끼리 점수가 0.02~0.03씩 흔들리므로, 차이가 0.003보다 작으면 "효과 없음"으로 판단함
- **결과(11/9, 김지연 실행)**
  - 재현 확인: LR 0.4204, 앙상블 0.4362 → v1 노트북 기록(LR 0.420, 편향 조정 전 0.436)과 같음
  - XGBoost: 0.4072 → **0.4205 (+0.013)**. 변이 표기 토큰을 안 쓰던 XGBoost에 위치 정보가 처음 들어가서 크게 오름
  - 앙상블: 0.4362 → **0.4417 (+0.0055)**
  - 아형별: ACC +0.041, LIHC +0.030, PAAD +0.025 상승 / GBMLGG −0.021 하락
  - LR에 핫스팟을 넣는 건 효과 없음 (따로 확인: 0.4148 → 0.4138). LR은 이미 `IDH1=R132H` 같은 표기 토큰을 쓰고 있어서 겹침
